# Reflection Patterns

**Companion lesson:** https://ml-viz.vercel.app/courses/agent-design-patterns/06-reflection-patterns

A from-scratch, runnable implementation of Self-Reflection, Cross-Reflection, and Human Reflection using mocked LLM responses — no API keys required.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
BRAND = '#6366f1'; TEAL = '#2dd4bf'; ROSE = '#fb7185'; YELLOW = '#fbbf24'
np.random.seed(0)

## Mocked LLM

We simulate a language model with deterministic, scripted responses. `MockLLM` takes a task string
and an `iteration` counter, and returns increasingly correct outputs as iterations increase —
mimicking the empirical observation that reflection loops converge toward correctness.

The mock injects a known error on `iteration=0` (the raw draft) and repairs it on later passes,
so we can trace exactly what the reflection loop is fixing.

In [ ]:
class MockLLM:
    """
    Deterministic mock of a language model.

    Responses are keyed by (task, iteration).  The error injected on
    iteration 0 (draft) is progressively corrected in later iterations.
    quality_score simulates a 0-1 quality metric that improves each round.
    """

    BUBBLE_SORT_RESPONSES = [
        # iteration 0: draft with deliberate errors
        (
            "Bubble sort works by repeatedly swapping adjacent elements that are in the "
            "wrong order. It has O(n) time complexity in all cases and is the fastest "
            "sorting algorithm for large datasets. The algorithm always makes exactly n "
            "passes through the array."
        ),
        # iteration 1: after self-critique — major errors fixed, minor issues remain
        (
            "Bubble sort works by repeatedly swapping adjacent elements that are in the "
            "wrong order. Its worst-case and average time complexity is O(n²). It is "
            "simple to implement but inefficient for large datasets. The number of passes "
            "can be fewer than n if an early-exit flag is used."
        ),
        # iteration 2: nearly correct
        (
            "Bubble sort repeatedly compares and swaps adjacent out-of-order elements. "
            "Worst-case and average time complexity is O(n²); best-case is O(n) with an "
            "early-exit optimisation. Space complexity is O(1). It is a stable sort, "
            "suitable for small or nearly-sorted datasets."
        ),
        # iteration 3: fully correct
        (
            "Bubble sort is a comparison-based sorting algorithm that repeatedly passes "
            "through the list, swapping adjacent elements that are out of order. "
            "Time complexity: O(n²) average and worst case, O(n) best case (optimised). "
            "Space complexity: O(1) in-place. It is a stable sort. Despite its simplicity, "
            "it is outperformed by insertion sort, merge sort, or Timsort in practice."
        ),
    ]

    BUBBLE_SORT_CRITIQUES = [
        # critique of iteration 0
        [
            "ERROR: Time complexity claim 'O(n)' is incorrect — bubble sort is O(n²) average/worst case.",
            "ERROR: 'fastest sorting algorithm for large datasets' is false — it is among the slowest.",
            "ERROR: 'always makes exactly n passes' is incorrect — with early exit, fewer passes are possible.",
        ],
        # critique of iteration 1
        [
            "MINOR: Best-case complexity (O(n) with early exit) not mentioned.",
            "MINOR: Stability property not stated.",
        ],
        # critique of iteration 2
        [
            "MINOR: Could compare explicitly to more practical alternatives (insertion sort, Timsort).",
        ],
        # critique of iteration 3
        [],  # no issues
    ]

    # Simulated quality scores per iteration (0–1)
    QUALITY_SCORES = [0.45, 0.72, 0.87, 0.95]

    def generate(self, task: str, iteration: int = 0) -> str:
        idx = min(iteration, len(self.BUBBLE_SORT_RESPONSES) - 1)
        return self.BUBBLE_SORT_RESPONSES[idx]

    def critique(self, draft: str, iteration: int = 0) -> list[str]:
        idx = min(iteration, len(self.BUBBLE_SORT_CRITIQUES) - 1)
        return self.BUBBLE_SORT_CRITIQUES[idx]

    def quality(self, iteration: int) -> float:
        idx = min(iteration, len(self.QUALITY_SCORES) - 1)
        return self.QUALITY_SCORES[idx]


llm = MockLLM()
print('MockLLM ready. Draft quality:', llm.quality(0))

## Self-Reflection

The `SelfReflection` class implements the generate → critique → revise loop.
A single agent plays both roles: generator and critic. After each critique it
revises the draft using the next scripted response (simulating incorporating the
critique). The loop stops when either the critique is empty or `max_iterations`
is reached.

We track the **quality score** and **critique list** at every iteration so we can
later plot the diminishing-returns curve.

In [ ]:
class SelfReflection:
    """
    Single-agent generate → critique → revise loop.

    Parameters
    ----------
    llm         : MockLLM (or any object with .generate, .critique, .quality)
    max_iters   : hard cap on the number of revision rounds
    """

    def __init__(self, llm: MockLLM, max_iters: int = 4):
        self.llm = llm
        self.max_iters = max_iters

    def run(self, task: str, verbose: bool = True) -> dict:
        history = []  # list of {iteration, draft, critique, quality}

        # --- Step 1: generate initial draft ---
        draft = self.llm.generate(task, iteration=0)
        quality = self.llm.quality(0)
        history.append({'iteration': 0, 'draft': draft,
                         'critique': None, 'quality': quality})

        if verbose:
            print(f'=== DRAFT (iteration 0) — quality {quality:.2f} ===')
            print(draft)
            print()

        for i in range(1, self.max_iters + 1):
            # --- Step 2: critique the current draft ---
            critique = self.llm.critique(draft, iteration=i - 1)

            if verbose:
                print(f'--- Critique after iteration {i - 1} ---')
                if critique:
                    for c in critique:
                        print(' •', c)
                else:
                    print(' (no issues found — stopping)')
                print()

            # --- Stopping condition: empty critique ---
            if not critique:
                break

            # --- Step 3: revise draft using critique ---
            draft = self.llm.generate(task, iteration=i)
            quality = self.llm.quality(i)
            history.append({'iteration': i, 'draft': draft,
                             'critique': critique, 'quality': quality})

            if verbose:
                print(f'=== REVISION {i} — quality {quality:.2f} ===')
                print(draft)
                print()

        return {'final': draft, 'history': history}


sr = SelfReflection(llm, max_iters=4)
result = sr.run('describe bubble sort')

## Cross-Reflection

`CrossReflection` uses two separate agent instances. Agent A drafts; Agent B
critiques; Agent A revises. We model Agent B with a second `MockLLM` instance
to show that the two agents are independent — in production they would be
separate model calls, potentially with different system prompts or temperatures.

The messages passed between agents are logged as the **inter-agent transcript**,
making the coordination overhead explicit.

In [ ]:
class CrossReflection:
    """
    Two-agent reflection: Agent A drafts, Agent B critiques, Agent A revises.

    Parameters
    ----------
    agent_a, agent_b : MockLLM instances (independent critics)
    max_rounds       : hard cap on review rounds
    """

    def __init__(self, agent_a: MockLLM, agent_b: MockLLM, max_rounds: int = 3):
        self.a = agent_a
        self.b = agent_b
        self.max_rounds = max_rounds

    def run(self, task: str, verbose: bool = True) -> dict:
        transcript = []  # messages between agents

        # Agent A: generate first draft
        draft = self.a.generate(task, iteration=0)
        transcript.append({'from': 'AgentA', 'to': 'AgentB',
                             'type': 'draft', 'content': draft})

        if verbose:
            print('=== Agent A → Agent B: DRAFT ===')
            print(draft)
            print()

        for round_idx in range(self.max_rounds):
            # Agent B: independently critique the draft
            critique = self.b.critique(draft, iteration=round_idx)
            transcript.append({'from': 'AgentB', 'to': 'AgentA',
                                 'type': 'critique', 'content': critique})

            if verbose:
                print(f'=== Agent B → Agent A: CRITIQUE (round {round_idx + 1}) ===')
                if critique:
                    for c in critique:
                        print(' •', c)
                else:
                    print(' (no issues found — Agent B approves)')
                print()

            if not critique:
                break

            # Agent A: revise based on Agent B's feedback
            draft = self.a.generate(task, iteration=round_idx + 1)
            transcript.append({'from': 'AgentA', 'to': 'AgentB',
                                 'type': 'revision', 'content': draft,
                                 'round': round_idx + 1})

            quality = self.a.quality(round_idx + 1)
            if verbose:
                print(f'=== Agent A → Agent B: REVISION {round_idx + 1} '
                      f'(quality {quality:.2f}) ===')
                print(draft)
                print()

        return {'final': draft, 'transcript': transcript}


agent_a = MockLLM()
agent_b = MockLLM()  # independent instance — different biases in a real system

cr = CrossReflection(agent_a, agent_b, max_rounds=3)
cr_result = cr.run('describe bubble sort')

print(f'Total messages exchanged between agents: {len(cr_result["transcript"])}')

## Quality vs iteration — the diminishing-returns curve

Each reflection iteration improves quality, but by less than the previous round.
We visualise this with the quality scores tracked during Self-Reflection, and
overlay the theoretical convergence curve from the lesson:

$$q_n \approx 1 - (1 - q_0)(1 - \alpha)^n$$

with $\alpha = 0.45$ (the per-iteration improvement rate).

In [ ]:
# Empirical scores from the Self-Reflection run
iterations = [h['iteration'] for h in result['history']]
qualities  = [h['quality']   for h in result['history']]

# Theoretical curve
q0, alpha = qualities[0], 0.45
n_theory = np.linspace(0, max(iterations), 200)
q_theory = 1 - (1 - q0) * (1 - alpha) ** n_theory

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: bar chart of per-iteration quality
ax = axes[0]
bars = ax.bar(iterations, qualities, color=BRAND, alpha=0.85, width=0.6)
ax.plot(n_theory, q_theory, color=TEAL, lw=2, ls='--', label='theoretical curve')
ax.set_xlabel('Iteration')
ax.set_ylabel('Quality score')
ax.set_title('Quality per iteration (Self-Reflection)')
ax.set_ylim(0, 1.05)
ax.set_xticks(iterations)
for bar, q in zip(bars, qualities):
    ax.text(bar.get_x() + bar.get_width() / 2, q + 0.02, f'{q:.2f}',
            ha='center', va='bottom', fontsize=10, color='#e2e8f0')
ax.legend()

# Right: per-iteration delta (diminishing returns)
ax2 = axes[1]
deltas = [0.0] + [qualities[i] - qualities[i-1] for i in range(1, len(qualities))]
ax2.bar(iterations, deltas, color=YELLOW, alpha=0.85, width=0.6)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('Δ quality')
ax2.set_title('Quality improvement per iteration (diminishing returns)')
ax2.set_xticks(iterations)
for i, (it, d) in enumerate(zip(iterations, deltas)):
    if d > 0:
        ax2.text(it, d + 0.005, f'+{d:.2f}',
                 ha='center', va='bottom', fontsize=10, color='#e2e8f0')

plt.tight_layout()
plt.show()

print(f'Quality gain: iteration 0→1: +{qualities[1]-qualities[0]:.2f}, '
      f'1→2: +{qualities[2]-qualities[1]:.2f}, '
      f'2→3: +{qualities[3]-qualities[2]:.2f}')
print('Most gain is captured in the first 1-2 iterations.')

## Comparison table

The three reflection patterns trade off latency, cost, independence, and human
overhead differently. The table below captures the key dimensions from the
lesson's comparison.

In [ ]:
patterns = [
    {
        'Pattern':             'None (single-pass)',
        'Latency multiplier':  '1×',
        'Cost multiplier':     '1×',
        'Error independence':  'None',
        'Human overhead':      'None',
        'Best use case':       'Low-stakes, reversible',
    },
    {
        'Pattern':             'Self-Reflection',
        'Latency multiplier':  '3–5×',
        'Cost multiplier':     '3–5×',
        'Error independence':  'Low (same model)',
        'Human overhead':      'None',
        'Best use case':       'High-stakes text/code generation',
    },
    {
        'Pattern':             'Cross-Reflection',
        'Latency multiplier':  '3–6×',
        'Cost multiplier':     '4–8×',
        'Error independence':  'High (different model)',
        'Human overhead':      'None',
        'Best use case':       'Adversarial, domain-bias-prone',
    },
    {
        'Pattern':             'Human Reflection',
        'Latency multiplier':  'Unbounded',
        'Cost multiplier':     'High (human time)',
        'Error independence':  'Complete',
        'Human overhead':      'Significant',
        'Best use case':       'Regulated / irreversible actions',
    },
]

cols = list(patterns[0].keys())
col_widths = {c: max(len(c), max(len(p[c]) for p in patterns)) for c in cols}

def row(p, cols, col_widths):
    return '  '.join(p[c].ljust(col_widths[c]) for c in cols)

header = row({c: c for c in cols}, cols, col_widths)
sep    = '  '.join('-' * col_widths[c] for c in cols)
print(header)
print(sep)
for p in patterns:
    print(row(p, cols, col_widths))

## ✏️ Your turn

**Challenge:** Implement `EarlyStopReflection` — a Self-Reflection loop that
stops not only when the critique is empty, but also when the **quality
improvement between iterations falls below a threshold `delta`**.

This mirrors a practical production requirement: once gains are marginal, there
is no point paying for another LLM call. The stopping criteria should be:

1. Critique is empty (existing condition), **OR**
2. $q_i - q_{i-1} < \delta$ (improvement below threshold), **OR**
3. `max_iters` reached (hard cap)

Your function should return `(final_draft, iterations_run, stop_reason)` where
`stop_reason` is one of `'empty_critique'`, `'delta_threshold'`, or `'max_iters'`.

In [ ]:
def early_stop_reflection(
    llm: MockLLM,
    task: str,
    delta: float = 0.10,
    max_iters: int = 4,
) -> tuple[str, int, str]:
    """
    Self-Reflection loop with delta-threshold early stopping.

    Returns
    -------
    (final_draft, iterations_run, stop_reason)
    stop_reason : 'empty_critique' | 'delta_threshold' | 'max_iters'
    """
    # TODO(you): generate initial draft and track its quality score.
    # Then loop: critique → check stopping conditions → revise.
    # Use llm.generate(task, iteration=i), llm.critique(draft, iteration=i),
    # and llm.quality(i).
    # Return (final_draft, iterations_run, stop_reason).
    pass


# --- tests (run after you implement the function) ---
draft, n_iters, reason = early_stop_reflection(llm, 'describe bubble sort', delta=0.10)

# Should stop before max_iters=4 because improvement drops below 0.10
assert draft is not None, 'Return a string draft'
assert isinstance(n_iters, int) and n_iters >= 1, 'n_iters must be a positive int'
assert reason in ('empty_critique', 'delta_threshold', 'max_iters'), \
    f'Unexpected stop_reason: {reason!r}'

# With delta=0.10, quality jumps are 0.27 (0→1), 0.15 (1→2), 0.08 (2→3)
# So the loop should stop at iteration 3 via delta_threshold (0.08 < 0.10)
assert reason == 'delta_threshold', f'Expected delta_threshold, got {reason!r}'
assert n_iters == 3, f'Expected 3 iterations, got {n_iters}'

print(f'passed ✓  stopped at iteration {n_iters} ({reason})')

<details><summary>Solution</summary>

```python
def early_stop_reflection(
    llm: MockLLM,
    task: str,
    delta: float = 0.10,
    max_iters: int = 4,
) -> tuple[str, int, str]:
    draft   = llm.generate(task, iteration=0)
    quality = llm.quality(0)

    for i in range(1, max_iters + 1):
        critique = llm.critique(draft, iteration=i - 1)

        if not critique:
            return draft, i - 1, 'empty_critique'

        new_draft   = llm.generate(task, iteration=i)
        new_quality = llm.quality(i)

        if new_quality - quality < delta:
            # Improvement is marginal — stop before applying this revision
            return new_draft, i, 'delta_threshold'

        draft, quality = new_draft, new_quality

    return draft, max_iters, 'max_iters'
```

</details>